# Exercises XP
## Data Preprocessing on the Titanic Dataset

This notebook follows the recommended preprocessing sequence:
1. Handle duplicates
2. Address missing values
3. Feature engineering
4. Treat outliers
5. Standardize / normalize
6. Encode categorical variables
7. Transform the Age feature into bins

All comments are in English.

In [ ]:
# Standard imports used throughout the notebook
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Display options
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

BASE_DIR = os.getcwd()
print('Working directory:', BASE_DIR)

In [ ]:
# Load the Titanic dataset (train.csv lives next to the notebook)
df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
print('Initial shape:', df.shape)
df.head()

## Exercise 1 — Duplicate Detection and Removal

In [ ]:
# 1) Detect duplicate rows over all columns
n_duplicates = df.duplicated().sum()
print(f'Number of duplicated rows (all columns): {n_duplicates}')

# Also check duplicates that ignore the unique PassengerId,
# in case two passengers were entered twice under different IDs.
n_dup_no_id = df.drop(columns=['PassengerId']).duplicated().sum()
print(f'Number of duplicated rows (ignoring PassengerId): {n_dup_no_id}')

In [ ]:
# 2) Remove duplicates and verify the row count before / after
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)

print(f'Rows before drop_duplicates(): {rows_before}')
print(f'Rows after  drop_duplicates(): {rows_after}')
print(f'Rows removed: {rows_before - rows_after}')

> **Comment:** The Titanic dataset is clean — no full-row duplicates were found. We still keep the `drop_duplicates()` call so the preprocessing pipeline is **robust** when reused on a less clean dataset.

## Exercise 2 : Handling Missing Values

In [ ]:
# 1) Identify columns with missing values and their percentage
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_count', ascending=False)
missing_report

### Strategy chosen for each column

| Column | Missing % | Strategy | Why |
|---|---|---|---|
| `Cabin` | ~77% | **Drop column** | Too many missing values to impute meaningfully. We will keep the *information* that a cabin was recorded as a binary feature `HasCabin`. |
| `Age` | ~20% | **Median imputation** | Numerical, skewed. Median is more robust than mean. We use `SimpleImputer(strategy='median')`. |
| `Embarked` | ~0.2% | **Mode (constant)** | Categorical with only 2 missing values → fill with the most frequent port. |

In [ ]:
# 2) Apply the strategies

# Cabin -> create a binary feature 'HasCabin' (1 if recorded, 0 otherwise), then drop the original column
df['HasCabin'] = df['Cabin'].notna().astype(int)
df = df.drop(columns=['Cabin'])

# Age -> median imputation with scikit-learn (works on a 2D array)
age_imputer = SimpleImputer(strategy='median')
df['Age'] = age_imputer.fit_transform(df[['Age']]).ravel()

# Embarked -> fill with the mode (most frequent value)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 3) Verify there are no missing values left
print('Missing values after imputation:')
print(df.isna().sum())

## Exercise 3 : Feature Engineering

In [ ]:
# 1) Family size = siblings/spouses + parents/children + the passenger himself
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Derived binary feature: IsAlone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

df[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

In [ ]:
# 2) Extract the title from the Name column (everything between ', ' and '. ')
#    Example: 'Braund, Mr. Owen Harris'  ->  'Mr'
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
print('Raw titles found:')
print(df['Title'].value_counts())

In [ ]:
# Group rare titles into a single 'Rare' category and normalize French / older variants
title_mapping = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare', 'the Countess': 'Rare'
}
df['Title'] = df['Title'].replace(title_mapping)
print('Cleaned titles:')
print(df['Title'].value_counts())

> Numerical features are **not** scaled yet — scaling happens after outlier handling (Exercise 5).  
> Encoding of the new `Title` column is also handled together with `Sex` and `Embarked` in Exercise 6.

## Exercise 4 : Outlier Detection and Handling

In [ ]:
# 1) Visual inspection of Age and Fare
fig, axes = plt.subplots(2, 2, figsize=(11, 6))

sns.boxplot(x=df['Age'],  ax=axes[0, 0], color='lightblue')
axes[0, 0].set_title('Boxplot — Age')

sns.histplot(df['Age'], bins=30, kde=True, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Histogram — Age')

sns.boxplot(x=df['Fare'], ax=axes[1, 0], color='salmon')
axes[1, 0].set_title('Boxplot — Fare')

sns.histplot(df['Fare'], bins=30, kde=True, ax=axes[1, 1], color='salmon')
axes[1, 1].set_title('Histogram — Fare')

plt.tight_layout()
plt.show()

In [ ]:
# 2a) IQR method on Fare
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_iqr, upper_iqr = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
iqr_outliers = ((df['Fare'] < lower_iqr) | (df['Fare'] > upper_iqr)).sum()
print(f'IQR method — Fare outliers: {iqr_outliers} (bounds: {lower_iqr:.2f} / {upper_iqr:.2f})')

# 2b) Z-score method on Fare
z_scores = (df['Fare'] - df['Fare'].mean()) / df['Fare'].std()
z_outliers = (z_scores.abs() > 3).sum()
print(f'Z-score method (|z| > 3) — Fare outliers: {z_outliers}')

In [ ]:
# 3) Handling strategies

# Keep a snapshot for the before/after comparison
fare_before = df['Fare'].copy()

# 3a) Quantile capping for Fare at the 98th percentile.
#     Rationale: the right tail of Fare is heavy. We do not want a 512$ outlier
#     to dominate scaling later. Capping at q=0.98 preserves the row but limits
#     the influence of the extreme value.
cap = df['Fare'].quantile(0.98)
df['Fare'] = df['Fare'].clip(upper=cap)
print(f'Fare capped at q=0.98 = {cap:.2f}')

# 3b) Log transformation for Fare to reduce skewness (log1p handles 0 values)
df['Fare_log'] = np.log1p(df['Fare'])

# 3c) Row removal is also an option — uncomment to drop rows above the cap instead
# df = df[df['Fare'] <= cap].reset_index(drop=True)

In [ ]:
# 4) Compare distributions before / after treatment
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
sns.histplot(fare_before, bins=30, kde=True, ax=axes[0], color='salmon')
axes[0].set_title('Fare — before')
sns.histplot(df['Fare'],  bins=30, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('Fare — after capping (q=0.98)')
sns.histplot(df['Fare_log'], bins=30, kde=True, ax=axes[2], color='seagreen')
axes[2].set_title('Fare — log1p transformation')
plt.tight_layout()
plt.show()

print('Before:', fare_before.describe().round(2).to_dict())
print('After :', df['Fare'].describe().round(2).to_dict())

## Exercise 5 : Standardization and Normalization

- `Age` is roughly bell-shaped after imputation → **StandardScaler** (z-score, mean = 0, std = 1).
- `Fare` is bounded and still right-skewed even after capping → **MinMaxScaler** (range [0, 1]).
- `FamilySize` is a small integer count → **MinMaxScaler** so it sits in [0, 1] like the others.

In [ ]:
# Standardize Age
std_scaler = StandardScaler()
df['Age_std'] = std_scaler.fit_transform(df[['Age']]).ravel()

# Normalize Fare and FamilySize to [0, 1]
mm_scaler = MinMaxScaler()
df[['Fare_mm', 'FamilySize_mm']] = mm_scaler.fit_transform(df[['Fare', 'FamilySize']])

df[['Age', 'Age_std', 'Fare', 'Fare_mm', 'FamilySize', 'FamilySize_mm']].describe().round(3)

## Exercise 6 : Feature Encoding

In [ ]:
# 1) Remaining categorical columns of interest
categorical_cols = ['Sex', 'Embarked', 'Title']
df[categorical_cols].head()

In [ ]:
# 2a) Sex has only two values -> we can use LabelEncoder for a single 0/1 column
le = LabelEncoder()
df['Sex_enc'] = le.fit_transform(df['Sex'])  # female -> 0, male -> 1 (alphabetical)
print('Sex encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# 2b) One-Hot Encoding for nominal variables (Embarked, Title)
df = pd.get_dummies(df, columns=['Embarked', 'Title'], prefix=['Embarked', 'Title'])

# 3) Quick look at the new dataframe schema
print('Shape after encoding:', df.shape)
print('Columns:')
print(list(df.columns))

## Exercise 7 : Data Transformation for the Age feature

In [ ]:
# 1) Build life-stage bins
bins   = [0, 12, 18, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']

df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, include_lowest=True)
df['AgeGroup'].value_counts()

In [ ]:
# 2) One-hot encode the age group with pd.get_dummies()
df = pd.get_dummies(df, columns=['AgeGroup'], prefix='AgeGroup')

# Preview the new columns
age_cols = [c for c in df.columns if c.startswith('AgeGroup_')]
df[['Age'] + age_cols].head()

## Final dataset preview

In [ ]:
print('Final shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

In [ ]:
# Save the preprocessed dataset so the next notebook can reuse it directly
out_path = os.path.join(BASE_DIR, 'train_preprocessed.csv')
df.to_csv(out_path, index=False)
print('Saved preprocessed dataset to:', out_path)


# Exercises XP Gold
## Scaling, composite features, normalization, reduction and aggregation

The Gold section reloads the original `train.csv` so it stays independent of
the heavy preprocessing pipeline above. It also uses two extra Kaggle datasets
already present in this folder:
- `superstore_dataset2011-2015.csv` — Superstore sales
- `city_day.csv` — Air Quality Data in India (daily measurements per city)

In [ ]:
# Reload a fresh copy of the Titanic dataset for the Gold exercises
titanic = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))

# Impute Age and Fare with their median so the scalers receive no NaN
titanic['Age']  = titanic['Age'].fillna(titanic['Age'].median())
titanic['Fare'] = titanic['Fare'].fillna(titanic['Fare'].median())

print('Shape:', titanic.shape)
titanic.head()

## Exercise 1 : Data Scaling and Normalization

In [ ]:
# Identify numerical columns of interest
numeric_cols = ['Age', 'Fare']
titanic[numeric_cols].describe().round(2)

In [ ]:
# Age is roughly bell-shaped -> Z-score standardization (StandardScaler)
# Fare is heavily skewed -> Min-Max normalization (MinMaxScaler) for a bounded [0,1] range
g1_std = StandardScaler()
g1_mm  = MinMaxScaler()

titanic['Age_std']  = g1_std.fit_transform(titanic[['Age']]).ravel()
titanic['Fare_mm']  = g1_mm.fit_transform(titanic[['Fare']]).ravel()

titanic[['Age', 'Age_std', 'Fare', 'Fare_mm']].describe().round(3)

### Effect of scaling on model performance
- **Distance-based models** (KNN, SVM with RBF kernel, K-Means) explicitly compute distances → without scaling, `Fare` (0–512) dominates `Age` (0–80) and the model effectively ignores `Age`.
- **Gradient-based models** (Logistic Regression, Neural Networks) converge much faster on scaled inputs because the loss surface becomes more isotropic.
- **Tree-based models** (Decision Tree, Random Forest, XGBoost) are *not* sensitive to scaling — splits are invariant to monotonic transformations.
- **Regularized models** (Ridge, Lasso) need scaling so that the penalty term treats every feature fairly.

## Exercise 2 : Creating Composite Features

In [ ]:
# Build the composite features
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic['IsAlone']    = (titanic['FamilySize'] == 1).astype(int)

titanic[['SibSp', 'Parch', 'FamilySize', 'IsAlone', 'Survived']].head()

In [ ]:
# Survival rate by family size
survival_by_family = titanic.groupby('FamilySize')['Survived'].mean().round(3)
print('Survival rate by FamilySize:')
print(survival_by_family)

# Survival rate by IsAlone
survival_by_alone = titanic.groupby('IsAlone')['Survived'].mean().round(3)
print('\nSurvival rate by IsAlone (1 = alone):')
print(survival_by_alone)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

survival_by_family.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Survival rate by Family Size')
axes[0].set_ylabel('Survival rate')
axes[0].set_xlabel('Family Size')
axes[0].axhline(titanic['Survived'].mean(), color='red', linestyle='--', label='Overall')
axes[0].legend()

survival_by_alone.plot(kind='bar', ax=axes[1], color=['#55a868', '#dd8452'], edgecolor='black')
axes[1].set_title('Survival rate — Alone vs With family')
axes[1].set_xticklabels(['With family', 'Alone'], rotation=0)
axes[1].set_ylabel('Survival rate')

plt.tight_layout()
plt.show()

## Exercise 3 : Min-Max and Z-score Normalization side by side

In [ ]:
# Apply both Min-Max normalization and Z-score normalization to Age and Fare
g3_mm  = MinMaxScaler()
g3_std = StandardScaler()

titanic[['Age_minmax',  'Fare_minmax']]  = g3_mm.fit_transform(titanic[['Age', 'Fare']])
titanic[['Age_zscore', 'Fare_zscore']]   = g3_std.fit_transform(titanic[['Age', 'Fare']])

titanic[['Age', 'Age_minmax', 'Age_zscore', 'Fare', 'Fare_minmax', 'Fare_zscore']].describe().round(3)

In [ ]:
# Compare histograms: original vs Min-Max vs Z-score for both Age and Fare
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

sns.histplot(titanic['Age'],         bins=30, kde=True, ax=axes[0, 0], color='steelblue'); axes[0, 0].set_title('Age — original')
sns.histplot(titanic['Age_minmax'],  bins=30, kde=True, ax=axes[0, 1], color='seagreen');  axes[0, 1].set_title('Age — Min-Max [0,1]')
sns.histplot(titanic['Age_zscore'],  bins=30, kde=True, ax=axes[0, 2], color='darkorange'); axes[0, 2].set_title('Age — Z-score')

sns.histplot(titanic['Fare'],         bins=30, kde=True, ax=axes[1, 0], color='salmon');     axes[1, 0].set_title('Fare — original')
sns.histplot(titanic['Fare_minmax'],  bins=30, kde=True, ax=axes[1, 1], color='seagreen');   axes[1, 1].set_title('Fare — Min-Max [0,1]')
sns.histplot(titanic['Fare_zscore'],  bins=30, kde=True, ax=axes[1, 2], color='darkorange'); axes[1, 2].set_title('Fare — Z-score')

plt.tight_layout()
plt.show()

## Exercise 4 : Data Reduction (PCA) and Aggregation

In [ ]:
# Principal Component Analysis on the numerical Titanic features
from sklearn.decomposition import PCA

features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']
X = titanic[features].copy()
X = StandardScaler().fit_transform(X)   # PCA requires scaled inputs

pca = PCA(n_components=2)
components = pca.fit_transform(X)

print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))
print('Cumulative explained variance:', pca.explained_variance_ratio_.cumsum().round(3))

In [ ]:
# Scatter plot of the two principal components, colored by survival
plt.figure(figsize=(8, 5))
plt.scatter(components[:, 0], components[:, 1], c=titanic['Survived'], cmap='coolwarm', alpha=0.6, edgecolor='k')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var.)')
plt.title('Titanic — first two principal components (color = survival)')
plt.colorbar(label='Survived')
plt.show()

In [ ]:
# Aggregate the data by Pclass and Embarked
agg = titanic.groupby(['Pclass', 'Embarked']).agg(
    n_passengers      = ('PassengerId', 'count'),
    mean_fare         = ('Fare',        'mean'),
    mean_age          = ('Age',         'mean'),
    survival_rate     = ('Survived',    'mean'),
).round(2)
agg

In [ ]:
# Heatmap of the survival rate per class / embarkation port
pivot = titanic.pivot_table(index='Pclass', columns='Embarked', values='Survived', aggfunc='mean').round(2)

plt.figure(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu', cbar_kws={'label': 'Survival rate'})
plt.title('Survival rate by Pclass and Embarked')
plt.show()

## Exercise 5 : Normalizing the Superstore Sales Dataset

In [ ]:
# The Superstore CSV uses ISO-8859-1 (latin-1) encoding
store_df = pd.read_csv(os.path.join(BASE_DIR, 'superstore_dataset2011-2015.csv'), encoding='latin-1')
print('Shape:', store_df.shape)
store_df[['Sales', 'Profit']].describe().round(2)

In [ ]:
# Min-Max normalization on Sales and Profit
scaler = MinMaxScaler()
store_df[['Sales_normalized', 'Profit_normalized']] = scaler.fit_transform(store_df[['Sales', 'Profit']])

store_df[['Sales', 'Sales_normalized', 'Profit', 'Profit_normalized']].head()

In [ ]:
# Sanity check: normalized columns should be in [0, 1]
print(store_df[['Sales_normalized', 'Profit_normalized']].describe().round(4))

## Exercise 6 : Aggregating the Air Quality dataset

In [ ]:
# Load the Air Quality dataset (city-level daily measurements in India)
air_df = pd.read_csv(os.path.join(BASE_DIR, 'city_day.csv'))
print('Shape:', air_df.shape)
air_df.head()

In [ ]:
# Convert the Date column to datetime and derive the year-month period
air_df['Date']  = pd.to_datetime(air_df['Date'])
air_df['Month'] = air_df['Date'].dt.to_period('M')

print('Date range:', air_df['Date'].min(), '->', air_df['Date'].max())
air_df[['City', 'Date', 'Month']].head()

In [ ]:
# Group by City and Month, then compute the monthly average of key pollutants
key_pollutants = ['PM2.5', 'PM10', 'NO2']
monthly_air = (
    air_df.groupby(['City', 'Month'])[key_pollutants]
          .mean()
          .round(2)
          .reset_index()
)
print('Aggregated shape:', monthly_air.shape)
monthly_air.head(10)

In [ ]:
# Visualize monthly PM2.5 trends for the 4 largest cities
top_cities = air_df['City'].value_counts().head(4).index.tolist()
subset = monthly_air[monthly_air['City'].isin(top_cities)].copy()
subset['Month'] = subset['Month'].dt.to_timestamp()

plt.figure(figsize=(11, 5))
sns.lineplot(data=subset, x='Month', y='PM2.5', hue='City', marker='o')
plt.title('Monthly average PM2.5 — top 4 cities')
plt.ylabel('PM2.5 (µg/m³)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()